In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:

# Import additional libraries
from sklearn.model_selection import train_test_split

# Convert categorical columns into numerical columns
dataset = pd.get_dummies(
    dataset,
    columns=['sex', 'smoker', 'region'],
    dtype=int
)

# Split the dataset into training and testing data
train_dataset, test_dataset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42
)

# Separate labels from features
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

# Convert data to float32
train_dataset = train_dataset.astype('float32')
test_dataset = test_dataset.astype('float32')

train_labels = train_labels.astype('float32')
test_labels = test_labels.astype('float32')

# Normalize input features
normalizer = layers.Normalization()
normalizer.adapt(np.array(train_dataset))

print("Training data:", train_dataset.shape)
print("Testing data:", test_dataset.shape)


In [ ]:

# Build the regression model
model = keras.Sequential([
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_absolute_error',
    metrics=['mae', 'mse']
)

# Stop training when validation loss stops improving
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    train_dataset,
    train_labels,
    epochs=300,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
